# CAM++ Model Conversion Comparison

This notebook compares different methods for converting the CAM++ ONNX model to PyTorch:
- ONNX Runtime (baseline)
- onnx2torch
- Custom model wrapper

It tests model conversion accuracy, output comparison, and gradient flow.

## Import Required Libraries

In [1]:
import sys
import torch
import numpy as np
import onnxruntime as ort
import torchaudio
import torchaudio.compliance.kaldi as kaldi
from pathlib import Path

## Define Preprocessing Function

In [2]:
def preprocess_audio(audio_path: str):
    """Preprocess audio exactly like generate_embedding.py does."""
    # 1. Load Audio
    speech, sample_rate = torchaudio.load(audio_path)
    
    # 2. Resample to 16000 Hz if necessary
    if sample_rate != 16000:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        speech = resampler(speech)
    
    # Ensure it's mono
    if speech.shape[0] > 1:
        speech = speech.mean(dim=0, keepdim=True)
        
    # 3. Extract Features (Fbank)
    feat = kaldi.fbank(speech,
                       num_mel_bins=80,
                       dither=0,
                       sample_frequency=16000)
    feat = feat - feat.mean(dim=0, keepdim=True)
    
    return feat

## Define Model Testing Functions

In [3]:
def test_onnx_runtime(onnx_path: str, input_data: np.ndarray):
    """Test with ONNX Runtime (baseline)."""
    print("\n=== Testing ONNX Runtime ===")
    
    option = ort.SessionOptions()
    option.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    option.intra_op_num_threads = 1
    
    session = ort.InferenceSession(onnx_path, sess_options=option, providers=["CPUExecutionProvider"])
    
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    
    print(f"Input: {input_name}, shape: {session.get_inputs()[0].shape}")
    print(f"Output: {output_name}, shape: {session.get_outputs()[0].shape}")
    
    output = session.run([output_name], {input_name: input_data})[0]
    
    print(f"Output shape: {output.shape}, output type: {output.dtype}")
    print(f"Output sample: {output[0, :5]}")
    
    return output

In [4]:
def test_onnx2torch(onnx_path: str, input_tensor: torch.Tensor):
    """Test with onnx2torch."""
    print("\n=== Testing onnx2torch ===")
    try:
        from onnx2torch import convert
        
        model = convert(onnx_path)
        
        # Introspect the model structure
        print("\n--- Model Structure Introspection ---")
        print(f"Model type: {type(model)}")
        print(f"Model training mode: {model.training}")
        
        # Find all BatchNorm layers
        bn_layers = []
        for name, module in model.named_modules():
            if isinstance(module, (torch.nn.BatchNorm1d, torch.nn.BatchNorm2d, torch.nn.BatchNorm3d)):
                bn_layers.append((name, module))
        
        if bn_layers:
            print(f"\nFound {len(bn_layers)} BatchNorm layers:")
            for name, module in bn_layers[:5]:  # Show first 5
                print(f"  {name}: {module}")
        
        # Try to inspect structure depth
        total_modules = sum(1 for _ in model.modules())
        print(f"Total modules in model: {total_modules}")
        
        # CRITICAL: Must use eval() mode with batch_size=1, otherwise BatchNorm fails
        # In training mode with batch_size=1, BatchNorm can't compute statistics
        print("\n⚠ Setting model to eval() mode (required for batch_size=1)")
        model.eval()
        
        print("\n--- Running Forward Pass ---")
        print(f"Input shape: {input_tensor.shape}")
        
        with torch.no_grad():
            output = model(input_tensor)
        
        output_np = output.cpu().numpy()
        print(f"✓ Conversion successful")
        print(f"Output shape: {output_np.shape}")
        print(f"Output sample: {output_np[0, :5]}")
        
        return output_np, model
        
    except Exception as e:
        import traceback
        print(f"✗ Failed: {e}")
        print("\n--- Full Traceback ---")
        traceback.print_exc()
        return None, None

In [5]:
def test_model_py(onnx_path: str, input_tensor: torch.Tensor):
    """Test using custom model wrapper."""
    print("\n=== Testing custom model ===")
    try:
        sys.path.append("./campplus_finetuning")
        from model import CampplusFinetunableModel
        model = CampplusFinetunableModel(onnx_path)
        model.train()
        model.unfreeze_all()
        model.freeze_batchnorm()  # Freeze BatchNorm for batch_size=1 training

        print("\n--- Running Forward Pass ---")
        print(f"Input shape: {input_tensor.shape}")
        with torch.no_grad():
            output = model(input_tensor)
        output_np = output.cpu().numpy()
        print(f"✓ Conversion successful")
        print(f"Output shape: {output_np.shape}")
        print(f"Output sample: {output_np[0, :5]}")
        return output_np, model
    except Exception as e:
        import traceback
        print(f"✗ Failed: {e}")
        print("\n--- Full Traceback ---")
        traceback.print_exc()
        return None, None

## Define Comparison and Analysis Functions

In [6]:
def compare_outputs(onnx_out, torch_out, pytorch_out, exported_out=None):
    """Compare outputs from all three methods."""
    print("\n=== Comparison Results ===")
    
    if torch_out is not None:
        diff_torch = np.abs(onnx_out - torch_out).max()
        print(f"ONNX vs onnx2torch max diff: {diff_torch:.6e}")
        print(f"  Are they close? {np.allclose(onnx_out, torch_out, atol=1e-5)}")
        # Calculate cosine similarity as well
        cos_sim = np.dot(onnx_out.flatten(), torch_out.flatten()) / (np.linalg.norm(onnx_out.flatten()) * np.linalg.norm(torch_out.flatten()))
        print(f"  Cosine similarity: {cos_sim:.6e}")
    
    if pytorch_out is not None:
        diff_pytorch = np.abs(onnx_out - pytorch_out).max()
        print(f"ONNX vs CampplusFinetunableMdoel max diff: {diff_pytorch:.6e}")
        print(f"  Are they close? {np.allclose(onnx_out, pytorch_out, atol=1e-5)}")
    
    if torch_out is not None and pytorch_out is not None:
        diff_both = np.abs(torch_out - pytorch_out).max()
        print(f"onnx2torch vs CampplusFinetunableMdoel max diff: {diff_both:.6e}")
    
    if exported_out is not None:
        diff_exported = np.abs(onnx_out - exported_out).max()
        print(f"ONNX vs exported ONNX max diff: {diff_exported:.6e}")
        print(f"  Are they close? {np.allclose(onnx_out, exported_out, atol=1e-5)}")

In [7]:
def test_gradient_flow(model, input_tensor: torch.Tensor):
    """Test if gradients flow through the model."""
    print("\n=== Testing Gradient Flow ===")
    
    if model is None:
        print("Model is None, skipping")
        return False
    
    input_tensor.requires_grad = True
    try:
        output = model(input_tensor)
    except Exception as e:
        import traceback
        print(f"✗ Failed: {e}")
        print("\n--- Full Traceback ---")
        traceback.print_exc()
        return None, None
    
    # Simple backward pass
    loss = output.sum()
    loss.backward()
    
    has_grad = input_tensor.grad is not None and input_tensor.grad.abs().sum() > 0
    print(f"Gradient flow: {'✓ Working' if has_grad else '✗ Not working'}")
    
    return has_grad

## Configuration

Set your model and audio file paths here:

In [8]:
# Configuration
audio_path = "./zero_shot_prompt.wav"
model_path = "./campplus.onnx"

print("=== CAM++ Model Conversion Comparison ===")
print(f"Model: {model_path}")
print(f"Audio: {audio_path}")

=== CAM++ Model Conversion Comparison ===
Model: ./campplus.onnx
Audio: ./zero_shot_prompt.wav


## Preprocess Audio

In [9]:
print("\n=== Preprocessing Audio ===")
feat = preprocess_audio(audio_path)
print(f"Feature shape: {feat.shape}")

# Prepare inputs for all three methods
# ONNX expects batch dimension
input_np = feat.unsqueeze(0).cpu().numpy()
input_torch = feat.unsqueeze(0)

print(f"Input shape for models: {input_np.shape}")


=== Preprocessing Audio ===
Feature shape: torch.Size([346, 80])
Input shape for models: (1, 346, 80)


## Test Model Conversions

In [10]:
# Test all three methods
onnx_output = test_onnx_runtime(model_path, input_np)
torch_output, torch_model = test_onnx2torch(model_path, input_torch)
custom_model_output, custom_model = test_model_py(model_path, input_torch)


=== Testing ONNX Runtime ===
Input: input, shape: ['batch_size', 'sequence_length', 80]
Output: output, shape: ['batch_size', 192]
Output shape: (1, 192), output type: float32
Output sample: [ 1.1294998   0.5331198  -0.591616   -0.61110085  1.3860643 ]

=== Testing onnx2torch ===

--- Model Structure Introspection ---
Model type: <class 'torch.fx.graph_module.GraphModule.__new__.<locals>.GraphModuleImpl'>
Model training mode: True

Found 55 BatchNorm layers:
  xvector/block1/tdnnd1/nonlinear1/batchnorm/BatchNormalization: BatchNorm1d(128, eps=9.999999747378752e-06, momentum=0.10000002384185791, affine=True, track_running_stats=True)
  xvector/block1/tdnnd2/nonlinear1/batchnorm/BatchNormalization: BatchNorm1d(160, eps=9.999999747378752e-06, momentum=0.10000002384185791, affine=True, track_running_stats=True)
  xvector/block1/tdnnd3/nonlinear1/batchnorm/BatchNormalization: BatchNorm1d(192, eps=9.999999747378752e-06, momentum=0.10000002384185791, affine=True, track_running_stats=True)
  

/home/john.zheng1/software/miniconda3/envs/wildtts-torchmetrics/lib/python3.10/site-packages/onnx2torch/node_converters/slice.py:63: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
  x = x[pos_axes_slices]


✓ Conversion successful
Output shape: (1, 192)
Output sample: [ 0.9833887  0.3727972 -0.6032647 -0.5367926  1.1683186]

=== Testing custom model ===

--- Running Forward Pass ---
Input shape: torch.Size([1, 346, 80])
✓ Conversion successful
Output shape: (1, 192)
Output sample: [ 0.9833887  0.3727972 -0.6032647 -0.5367926  1.1683186]


## Test Model Export

In [11]:
sys.path.append('./campplus_finetuning')
from train_simple import export_onnx_model

# Test exporting the original model
exported_path = "./test_exported.onnx"
export_onnx_model(custom_model, exported_path)

exported_output = test_onnx_runtime(exported_path, input_np)

/home/john.zheng1/software/miniconda3/envs/wildtts-torchmetrics/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/john.zheng1/software/miniconda3/envs/wildtts-torchmetrics/lib/python3.10/site-packages/onnx2torch/node_converters/shape.py:44: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  return torch.tensor(
/home/john.zheng1/software/miniconda3/envs/wildtts-torchmetrics/lib/python3.10/site-packages/onnx2torch/node_converters/reshape.py:20: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't recor

Model exported to ./test_exported.onnx

=== Testing ONNX Runtime ===
Input: speech, shape: ['batch_size', 'time', 80]
Output: embedding, shape: ['batch_size', 192]
Output shape: (1, 192), output type: float32
Output sample: [ 1.1294998   0.5331198  -0.591616   -0.61110085  1.3860643 ]


## Compare Outputs

In [12]:
compare_outputs(onnx_output, torch_output, custom_model_output, exported_output)


=== Comparison Results ===
ONNX vs onnx2torch max diff: 4.601316e-01
  Are they close? False
  Cosine similarity: 9.924982e-01
ONNX vs CampplusFinetunableMdoel max diff: 4.601316e-01
  Are they close? False
onnx2torch vs CampplusFinetunableMdoel max diff: 0.000000e+00
ONNX vs exported ONNX max diff: 0.000000e+00
  Are they close? True


## Test Gradient Flow

In [13]:
if torch_model is not None:
    print("\n--- onnx2torch gradient test ---")
    test_gradient_flow(torch_model, input_torch.clone())


--- onnx2torch gradient test ---

=== Testing Gradient Flow ===


Gradient flow: ✓ Working


In [14]:
if custom_model is not None:
    print("\n--- CampplusFinetunableMdoel gradient test ---")
    test_gradient_flow(custom_model, input_torch.clone())


--- CampplusFinetunableMdoel gradient test ---

=== Testing Gradient Flow ===
Gradient flow: ✓ Working


## Recommendations

In [15]:
print("\n=== Recommendation ===")
if torch_output is not None and np.allclose(onnx_output, torch_output, atol=1e-5):
    print("✓ onnx2torch: Working correctly")
if custom_model_output is not None and np.allclose(onnx_output, custom_model_output, atol=1e-5):
    print("✓ CampplusFinetunableMdoel: Working correctly")


=== Recommendation ===
